# Contenu
- Chargement et nettoyage des données
- Statistiques descriptives interactives
- Analyse des substances (graphiques dynamiques)
- Analyse des titulaires
- Analyse spatiale interactive
- Évolution temporelle avec curseur
- Relations entre variables
- Tableaux de bord interactifs

In [ ]:

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re
from collections import Counter
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display, HTML

## 1. Chargement des données

In [ ]:
file_path = './donnees-tabulaires_memoire.xls'
df = pd.read_excel(file_path, sheet_name='Feuil1')

# Nettoyage des noms de colonnes
df.columns = df.columns.str.strip()

print(f"Nombre de permis chargés : {len(df)}")
df.head()

Nombre de permis chargés : 5003


,N°permis,Type,Titulaire,Classification,Nombre carrés,Substances,Date d'octroi initial,Date fin de validité initiale,Dernier FA payé,EN_COURS,Localisation
0,1,E,GALLOIS Etablissement,REGULIER,64,-Graphite,1999-11-08,2029-11-07,2020,NaN,Amboditavolo (21) / Sahamatevina (43)
1,4,E,SOCIETE MALGACHE DU GRAPHITE S.A.,EN ATTENTE DE DECISION D'ANNULATION,16,-Graphite,2001-10-11,2041-10-10,2018,NaN,Andranobolaha (16)
2,5,E,GALLOIS Etablissement,REGULIER,48,-Graphite,1999-11-08,2029-11-07,2020,NaN,Andranobolaha (7) / Anjahamana (41)
3,13,E,SOMIDA S.A.,EN ATTENTE DE DECISION D'ANNULATION,16,-Mica,1999-10-21,2039-10-20,2018,NaN,Ranopiso (16)
4,19,E,GALLOIS Etablissement,REGULIER,32,-Graphite,1999-11-08,2036-03-08,2020,NaN,Ambinaninony (16) / Ampasimadinika Manambolo (16)


## 2. Prétraitement des données

In [ ]:
# Conversion des dates
date_cols = ['Date d\'octroi initial', 'Date fin de validité initiale']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Extraction de l'année d'octroi
df['Annee_octroi'] = df['Date d\'octroi initial'].dt.year

# Extraction des substances
def split_substances(s):
    if pd.isna(s):
        return []
    s = s.strip()
    if s.startswith('-'):
        s = s[1:]
    items = re.split(r'\s*-\s*', s)
    return [item.strip() for item in items if item.strip()]

df['Substances_list'] = df['Substances'].apply(split_substances)
df['Nb_substances'] = df['Substances_list'].apply(len)

# Comptage des localisations
def count_locations(loc_str):
    if pd.isna(loc_str):
        return 0
    return loc_str.count('(')

df['Nb_localisations'] = df['Localisation'].apply(count_locations)

# Classification des titulaires
def classify_titulaire(name):
    if isinstance(name, str):
        if any(key in name for key in ['S.A.', 'S.A.R.L.', 'SARL', 'S.A.R.L.U.', 'Ltd', 'S.A.R.L.U', 'S.A.U', 'Co.', 'Corporation', 'Group', 'Company', 'S.A']):
            return 'Société'
        else:
            return 'Personne physique'
    return 'Inconnu'

df['Type_titulaire'] = df['Titulaire'].apply(classify_titulaire)

# Extraction des événements
def extract_events(desc):
    if pd.isna(desc):
        return []
    keywords = ['RENOUVELLEMENT', 'CESSION', 'TRANSFORMATION', 'EXTENSION', 'ANNULATION', 'PARTENARIAT', 'DESISTEMENT', 'OCTROI']
    return [kw for kw in keywords if kw in desc.upper()]

df['Evenements'] = df['EN_COURS'].apply(extract_events)
df['Nb_evenements'] = df['Evenements'].apply(len)

# Extraire les localisations
def extract_locations(loc_str):
    if pd.isna(loc_str):
        return []
    parts = re.split(r'\s*\(\d+\)\s*', loc_str)
    return [p.strip() for p in parts if p.strip()]

df['Localisations_list'] = df['Localisation'].apply(extract_locations)

print("Prétraitement terminé")
df[['N°permis', 'Type', 'Classification', 'Nb_substances', 'Nb_localisations', 'Annee_octroi']].head()

Prétraitement terminé


,N°permis,Type,Classification,Nb_substances,Nb_localisations,Annee_octroi
0,1,E,REGULIER,1,2,1999.0
1,4,E,EN ATTENTE DE DECISION D'ANNULATION,1,1,2001.0
2,5,E,REGULIER,1,2,1999.0
3,13,E,EN ATTENTE DE DECISION D'ANNULATION,1,1,1999.0
4,19,E,REGULIER,1,2,1999.0


## 3. Statistiques descriptives interactives

In [ ]:
# 3.1 Répartition par type de permis - Graphique à barres interactif
type_counts = df['Type'].value_counts().reset_index()
type_counts.columns = ['Type', 'Nombre']

fig1 = px.bar(type_counts, x='Type', y='Nombre',
              title='Répartition par type de permis',
              color='Type',
              color_discrete_sequence=px.colors.qualitative.Set2,
              text='Nombre')
fig1.update_traces(textposition='outside')
fig1.show()

In [ ]:
# 3.2 Classification administrative - Graphique à barres interactif
class_counts = df['Classification'].value_counts().reset_index()
class_counts.columns = ['Classification', 'Nombre']

fig2 = px.bar(class_counts, x='Classification', y='Nombre',
              title='Classification des permis',
              color='Classification',
              color_discrete_sequence=px.colors.qualitative.Pastel,
              text='Nombre')
fig2.update_traces(textposition='outside')
fig2.update_layout(xaxis_tickangle=-45)
fig2.show()

In [ ]:
# 3.3 Distribution des superficies - Histogramme interactif
fig3 = px.histogram(df, x='Nombre carrés',
                    title='Distribution des superficies (Nombre carrés)',
                    nbins=50,
                    color_discrete_sequence=['#2E86C1'])
fig3.update_layout(xaxis_range=[0, 500])
fig3.show()

In [ ]:
# 3.4 Statistiques de superficie - Boîte à moustaches interactive
fig4 = px.box(df, x='Type', y='Nombre carrés',
              title='Distribution des superficies par type de permis',
              color='Type',
              log_y=True,
              color_discrete_sequence=px.colors.qualitative.Set1)
fig4.show()

## 4. Analyse des substances

In [ ]:
# 4.1 Top 20 des substances les plus fréquentes
all_substances = []
for sub_list in df['Substances_list']:
    all_substances.extend(sub_list)

sub_counter = Counter(all_substances)
top_substances = pd.DataFrame(sub_counter.most_common(20), columns=['Substance', 'Fréquence'])

fig5 = px.bar(top_substances, x='Fréquence', y='Substance',
              title='Top 20 des substances les plus fréquentes',
              orientation='h',
              color='Fréquence',
              color_continuous_scale='Viridis',
              text='Fréquence')
fig5.update_traces(textposition='outside')
fig5.show()

In [ ]:
# 4.2 Nombre de substances par permis
fig6 = px.histogram(df, x='Nb_substances',
                    title='Distribution du nombre de substances par permis',
                    nbins=20,
                    color_discrete_sequence=['#E74C3C'])
fig6.show()

In [ ]:
# 4.3 Carte de chaleur des substances par type de permis
top10_sub = [s for s, _ in sub_counter.most_common(10)]

for sub in top10_sub:
    df[sub] = df['Substances_list'].apply(lambda x: 1 if sub in x else 0)

heatmap_data = df.groupby('Type')[top10_sub].mean().reset_index()
heatmap_melt = heatmap_data.melt(id_vars='Type', var_name='Substance', value_name='Proportion')

fig7 = px.density_heatmap(heatmap_melt, x='Substance', y='Type', z='Proportion',
                          title='Proportion des substances les plus fréquentes par type de permis',
                          color_continuous_scale='YlOrRd',
                          text_auto='.2f')
fig7.show()

## 5. Analyse des titulaires

In [ ]:
# 5.1 Top 15 titulaires
top_titulaires = df['Titulaire'].value_counts().head(15).reset_index()
top_titulaires.columns = ['Titulaire', 'Nombre']

fig8 = px.bar(top_titulaires, x='Nombre', y='Titulaire',
              title='Top 15 titulaires par nombre de permis',
              orientation='h',
              color='Nombre',
              color_continuous_scale='Portland',
              text='Nombre')
fig8.update_traces(textposition='outside')
fig8.show()

In [ ]:
# 5.2 Répartition des titulaires (Société vs Personne physique)
type_tit_counts = df['Type_titulaire'].value_counts().reset_index()
type_tit_counts.columns = ['Type', 'Nombre']

fig9 = px.pie(type_tit_counts, values='Nombre', names='Type',
              title='Répartition des titulaires (Société vs Personne physique)',
              color_discrete_sequence=px.colors.qualitative.Set2,
              hole=0.3)
fig9.show()

## 6. Analyse spatiale interactive

In [ ]:
# Extraire toutes les localisations
all_locations = []
for loc in df['Localisation']:
    if pd.notna(loc):
        all_locations.extend(extract_locations(loc))

loc_counter = Counter(all_locations)
top_locations = pd.DataFrame(loc_counter.most_common(20), columns=['Localité', 'Fréquence'])

fig10 = px.bar(top_locations, x='Fréquence', y='Localité',
               title='Top 20 des localités les plus citées',
               orientation='h',
               color='Fréquence',
               color_continuous_scale='Blues',
               text='Fréquence')
fig10.update_traces(textposition='outside')
fig10.show()

In [ ]:
# 6.2 Carte des localisations (simulée avec scatter plot)
# On utilise ici une visualisation alternative avec le nombre de localisations par permis

fig11 = px.scatter(df, x='Nb_localisations', y='Nb_substances',
                   color='Type',
                   size='Nombre carrés',
                   hover_data=['Titulaire', 'Classification'],
                   title='Relation entre nombre de localisations et nombre de substances',
                   labels={'Nb_localisations': 'Nombre de localités',
                          'Nb_substances': 'Nombre de substances'},
                   log_x=True,
                   size_max=40)
fig11.show()

## 7. Évolution temporelle

In [ ]:
# 7.1 Évolution des octrois par année
year_counts = df['Annee_octroi'].value_counts().sort_index().reset_index()
year_counts.columns = ['Année', 'Nombre']

fig12 = px.bar(year_counts, x='Année', y='Nombre',
               title="Nombre d'octrois par année",
               color='Nombre',
               color_continuous_scale='Teal',
               text='Nombre')
fig12.update_traces(textposition='outside')
fig12.show()

In [ ]:
# 7.2 Évolution des octrois par type (graphique en aires)
pivot_type_year = pd.crosstab(df['Annee_octroi'], df['Type']).reset_index()
pivot_type_year_melt = pivot_type_year.melt(id_vars='Annee_octroi', var_name='Type', value_name='Nombre')

fig13 = px.area(pivot_type_year_melt, x='Annee_octroi', y='Nombre', color='Type',
                title='Évolution des octrois par type de permis',
                color_discrete_sequence=px.colors.qualitative.Set1)
fig13.show()

In [ ]:
# 7.3 Animation temporelle (permis octroyés par année)
df_year = df[df['Annee_octroi'].notna()].copy()
df_year['Année_str'] = df_year['Annee_octroi'].astype(int).astype(str)

fig14 = px.scatter(df_year, x='Nb_substances', y='Nombre carrés',
                   animation_frame='Année_str',
                   color='Type',
                   hover_name='Titulaire',
                   title='Évolution des permis dans le temps',
                   labels={'Nb_substances': 'Nombre de substances',
                          'Nombre carrés': 'Superficie'},
                   log_y=True,
                   size='Nombre carrés',
                   size_max=30,
                   range_x=[0, 20],
                   range_y=[0.5, 10000])
fig14.show()

## 8. Événements administratifs

In [ ]:
# Compter les événements
all_events = []
for ev_list in df['Evenements']:
    all_events.extend(ev_list)

event_counts = pd.DataFrame(Counter(all_events).items(), columns=['Événement', 'Fréquence'])

fig15 = px.bar(event_counts, x='Fréquence', y='Événement',
               title='Fréquence des événements administratifs',
               orientation='h',
               color='Fréquence',
               color_continuous_scale='Purples',
               text='Fréquence')
fig15.update_traces(textposition='outside')
fig15.show()

In [ ]:
# Relation entre événements et classification
event_class = df[df['Nb_evenements'] > 0].groupby('Classification')['Nb_evenements'].mean().reset_index()

fig16 = px.bar(event_class, x='Classification', y='Nb_evenements',
               title='Nombre moyen d\'événements par classification',
               color='Nb_evenements',
               color_continuous_scale='Oranges',
               text='Nb_evenements')
fig16.update_traces(textposition='outside')
fig16.show()

## 9. Relations avancées

In [ ]:
# 9.1 Matrice de corrélation des variables numériques
numeric_cols = ['Nombre carrés', 'Nb_substances', 'Nb_localisations', 'Nb_evenements']
corr_matrix = df[numeric_cols].corr()

fig17 = px.imshow(corr_matrix,
                  title='Matrice de corrélation des variables numériques',
                  color_continuous_scale='RdBu_r',
                  text_auto='.2f')
fig17.show()

In [ ]:
# 9.2 Scatter matrix interactif
fig18 = px.scatter_matrix(df,
                          dimensions=['Nombre carrés', 'Nb_substances', 'Nb_localisations', 'Annee_octroi'],
                          color='Type',
                          title='Matrice de dispersion des variables principales',
                          height=800)
fig18.show()

## 10. Tableau de bord interactif avec sous-graphiques

In [ ]:
# Création d'un tableau de bord avec 4 graphiques
fig19 = make_subplots(rows=2, cols=2,
                      subplot_titles=('Type de permis', 'Classification',
                                     'Top 10 substances', 'Évolution des octrois'),
                      specs=[[{'type': 'pie'}, {'type': 'bar'}],
                             [{'type': 'bar'}, {'type': 'scatter'}]])

# 1. Type de permis (pie)
type_pie = df['Type'].value_counts()
fig19.add_trace(go.Pie(labels=type_pie.index, values=type_pie.values, hole=0.3), row=1, col=1)

# 2. Classification (bar)
class_bar = df['Classification'].value_counts().head(5)
fig19.add_trace(go.Bar(x=class_bar.index, y=class_bar.values, marker_color='coral'), row=1, col=2)

# 3. Top 10 substances (bar)
top10 = pd.DataFrame(sub_counter.most_common(10), columns=['Substance', 'Fréquence'])
fig19.add_trace(go.Bar(x=top10['Fréquence'], y=top10['Substance'], orientation='h', marker_color='teal'), row=2, col=1)

# 4. Évolution des octrois (scatter)
year_counts_filtered = year_counts[year_counts['Année'].notna()]
fig19.add_trace(go.Scatter(x=year_counts_filtered['Année'], y=year_counts_filtered['Nombre'],
                          mode='lines+markers', marker_color='navy'), row=2, col=2)

fig19.update_layout(title='Tableau de bord interactif - Analyse minière', height=800, showlegend=False)
fig19.show()

## 11. Filtres interactifs avec Dropdown

In [ ]:
# Création d'un graphique avec filtre interactif par type de substance
# Sélection des 5 substances les plus fréquentes
top5_sub = [s for s, _ in sub_counter.most_common(5)]

# Création d'un graphique à barres groupées
df_sub_year = df.copy()
for sub in top5_sub:
    df_sub_year[sub] = df_sub_year['Substances_list'].apply(lambda x: 1 if sub in x else 0)

sub_year_pivot = df_sub_year.groupby('Annee_octroi')[top5_sub].sum().reset_index()
sub_year_melt = sub_year_pivot.melt(id_vars='Annee_octroi', var_name='Substance', value_name='Nombre')

fig20 = px.line(sub_year_melt, x='Annee_octroi', y='Nombre', color='Substance',
                title='Évolution des 5 principales substances par année',
                color_discrete_sequence=px.colors.qualitative.Set2,
                markers=True)
fig20.show()

**Interpretation :**
- **74 %** des permis sont des permis d'exploitation (type E)
- **35 %** des permis sont classés comme "Régulier"
- **L'Or, le Béryl, le Cristal, la Tourmaline et le Graphite** sont les substances les plus fréquentes
- **PAM Madagascar S.A** est le titulaire le plus actif
- Les localités les plus citées sont **Andranobolaha, Manakana, et Ilakaka**
- **Une forte activité de renouvellement et de cession** est observée
- Les octrois ont connu des pics en **1999, 2001, 2015 et 2016**
